In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, f1_score
from utils.preprocessing import load_data, split_features_target, scale_data, data_split, remove_zero_columns2
from utils.user_utils import get_model_train_eval


In [ ]:
# 데이터 로딩 및 기본 전처리
train, test = load_data()
X_features, y_labels = split_features_target(train)
X_test = test.drop(columns=['ID'], axis=1)

# zero_count_rate 제거
X_features, X_test = remove_zero_columns2(X_features, X_test, 0.99)

# var3 처리
X_features['var3'] = X_features['var3'].replace(-999999, 2)

# threshold 리스트
thresholds = [0.85, 0.90, 0.95]

results = []

for th in thresholds:
    # 1. 상관계수 행렬 계산
    corr_matrix = X_features.corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    
    # 2. threshold 이상인 컬럼 drop
    to_drop = [column for column in upper.columns if any(upper[column] > th)]
    X_reduced = X_features.drop(columns=to_drop)
    X_test_reduced = X_test.drop(columns=to_drop)
    
    # 3. 스케일링
    X_train_scaled, X_test_scaled, scaler = scale_data(X_reduced, X_test_reduced)
    
    # 4. 학습/검증 데이터 분리
    X_train, X_val, y_train, y_val = data_split(X_reduced, y_labels)
    
    # 5. 모델 학습
    rf_clf = RandomForestClassifier(
        random_state=0,
        n_estimators=390,
        max_depth=25,
        class_weight={0:1, 1:2},
        min_samples_leaf=1,
        min_samples_split=7,
        n_jobs=-1
    )
    rf_clf.fit(X_train, y_train)
    
    # 6. 성능 평가
    y_pred = rf_clf.predict(X_val)
    y_proba = rf_clf.predict_proba(X_val)[:,1]
    
    auc = roc_auc_score(y_val, y_proba)
    f1 = f1_score(y_val, y_pred)
    
    results.append({
        "threshold": th,
        "dropped_features": len(to_drop),
        "AUC": auc,
        "F1": f1
    })

# 결과 출력
results_df = pd.DataFrame(results)
print(results_df)
